In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid
from datetime import datetime

In [0]:
storage_account = "medallionhealthdata03"       
access_key      = "PvqLeriZR/6Wau/Xj/Al+FNEtT/D/+sRbeTdTYildN+qiN1SDuM4sG6hTLrkPGAcw9nV5B7t9bvs+AStj7fPGw=="
 
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    access_key
)
 

In [0]:
landing_path = f"abfss://landing@{storage_account}.dfs.core.windows.net/"
bronze_path  = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
silver_path  = f"abfss://silver@{storage_account}.dfs.core.windows.net/"
gold_path    = f"abfss://gold@{storage_account}.dfs.core.windows.net/"
 
spark.sql("CREATE DATABASE IF NOT EXISTS medallion_healthcare")
spark.sql("USE medallion_healthcare")
 
print("Connected. Database ready.")
display(dbutils.fs.ls(landing_path))

Connected. Database ready.


path,name,size,modificationTime
abfss://landing@medallionhealthdata03.dfs.core.windows.net/appointments.csv,appointments.csv,10907,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/billing.csv,billing.csv,10018,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/doctors.csv,doctors.csv,962,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/patients.csv,patients.csv,5626,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/treatments.csv,treatments.csv,11072,1783938174000


In [0]:

batch_id = str(uuid.uuid4())
pipeline_version = "1.0"
run_timestamp = datetime.now()

In [0]:

print("Batch ID :", batch_id)
print("Pipeline Version :", pipeline_version)
print("Run Time :", run_timestamp)
     

Batch ID : 466a8dec-edf9-44f4-91e6-7d8910457843
Pipeline Version : 1.0
Run Time : 2026-07-13 10:26:23.177296


# Setting metadata

In [0]:
metadata = [
("patients",
 "patients.csv",
 "bronze_patients",
 "Patient_ID",
 "FULL",
 "Y"),
("doctors",
 "doctors.csv",
 "bronze_doctors",
 "Doctor_ID",
 "FULL",
 "Y"),
("appointments",
 "appointments.csv",
 "bronze_appointments",
 "Appointment_ID",
 "FULL",
 "Y"),
("billing",
 "billing.csv",
 "bronze_billing",
 "Billing_ID",
 "FULL",
 "Y"),
("treatments",
 "treatments.csv",
 "bronze_treatments",
 "Treatment_ID",
 "FULL",
 "Y")
]
     

In [0]:

metadata_schema = StructType([
    StructField("source_name", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("primary_key", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("active_flag", StringType(), True)
])

In [0]:

metadata_df = spark.createDataFrame(metadata, metadata_schema)
     

In [0]:
display(metadata_df)

source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


In [0]:
metadata_df.write \
.mode("overwrite") \
.format("delta") \
.save(f"{bronze_path}metadata_config")
     

In [0]:
display(
    spark.read.format("delta")
    .load(f"{bronze_path}metadata_config")
)

source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


In [0]:
audit_schema = StructType([
StructField("batch_id", StringType(), True),
StructField("source_name", StringType(), True),
StructField("layer", StringType(), True),
StructField("rows_read", IntegerType(), True),
StructField("rows_written", IntegerType(), True),
StructField("status", StringType(), True),
StructField("start_time", TimestampType(), True),
StructField("end_time", TimestampType(), True),
StructField("error_message", StringType(), True)
])

In [0]:
audit_df = spark.createDataFrame([], audit_schema)

In [0]:
audit_df.write \
.mode("overwrite") \
.format("delta") \
.save(f"{bronze_path}audit_log")

In [0]:
display(
    spark.read.format("delta")
    .load(f"{bronze_path}audit_log")
)

batch_id,source_name,layer,rows_read,rows_written,status,start_time,end_time,error_message
